In [1]:
#!pip install openpyxl

In [2]:
#pip install -r requirements.txt

# Zusammenfassung Ergebnisse Notebook
* Data Cleaning von Soll- und Ist-Daten -> csv Export für weitere Arbeit (mergen ging nicht aufgrund von fehlerhaften Daten und unterschiedlicher Zeilenanzahl und Stoppreihenfolge)
    * fügen allen Ist-Daten Tournummern zu anhand von Fahrer, Kennzeichen und Datum
    * Analyse fehlender Touren
    * manche Fahrer und Kennzeichen von den Soll-Daten sind gar nicht in den Ist-Daten
    * Formularbeschreibung ("Ankunft/Abfahrt Kunde") entfernt und stattdessen Meldungszeit in Spalten `Abfahrt` und `Ankunft` (Tabelle teilweise pivotisieren)

# Import Packages

In [3]:
import pandas as pd
import numpy as np

# Soll Daten einlesen aus CIS-Daten.xlsx 

In [4]:
# Nur die gewünschten Spalten einlesen
cols = ['TOURNR', 'LKW_KENNZ', 'FAHRERNAME', 'TOUR', 'ANKUNFT', 'ABFAHRT', 'GEOX', 'GEOY']

df_soll = pd.read_excel('CIS-Daten.xlsx', usecols=cols)

df_soll.head()

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 06:00,02.03.2026 06:00,9.98632,53.52348
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 06:00,02.03.2026 06:30,9.98632,53.52348
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 06:49,02.03.2026 07:19,9.99383,53.54751
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 07:41,02.03.2026 08:11,9.98632,53.52348
4,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 08:31,02.03.2026 09:01,9.99383,53.54751


Anzahl aller Zeilen mit einer Tournummer -> alle Zeilen der Soll-Daten haben eine Tournummer

In [5]:
# Anzahl aller Zeilen mit einer Tournummer (inkl. Duplikate)
print("alle zeilen: ", df_soll.shape)
print(df_soll['TOURNR'].count())

alle zeilen:  (2800, 8)
2800


Schauen, ob es Duplikate gibt in den Zeilen -> gibt es nur für Mafi Greif Fahrzeuge und die sollen laut Firma sowieso gelöscht werden

In [6]:
df_soll.duplicated().any()

np.True_

In [7]:
df_soll[df_soll.duplicated(keep=False)]

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY
1439,H/42521,Mafi Greif,"Stoffer, Andreas",H/42521,19.03.2026 07:20,19.03.2026 07:40,9.98632,53.52348
1440,H/42521,Mafi Greif,"Stoffer, Andreas",H/42521,19.03.2026 07:20,19.03.2026 07:40,9.98632,53.52348
1441,H/42521,Mafi Greif,"Stoffer, Andreas",H/42521,19.03.2026 08:00,19.03.2026 08:20,9.98632,53.52348
1442,H/42521,Mafi Greif,"Stoffer, Andreas",H/42521,19.03.2026 08:00,19.03.2026 08:20,9.98632,53.52348


## Häufigkeit jeder Tournummer

In [8]:
# Häufigkeit jeder Tournummer
print(df_soll['TOURNR'].value_counts())

TOURNR
H/42519    77
H/42518    50
H/42444    47
H/42520    40
H/42441    39
           ..
H/42446     3
H/42638     3
H/42605     3
H/42606     3
H/42675     3
Name: count, Length: 265, dtype: int64


erste Tour hat 77 Stopps -> gucken da genauer rein -> Mafi Greif Fahrzeug, wird gelöscht

In [9]:
# Alle Einträge für Tournummer H/42519
df_tour = df_soll[df_soll['TOURNR'] == 'H/42519']

print(df_tour)

       TOURNR   LKW_KENNZ        FAHRERNAME     TOUR           ANKUNFT  \
1318  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 06:00   
1319  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 06:00   
1320  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 06:25   
1321  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 06:45   
1322  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 07:05   
...       ...         ...               ...      ...               ...   
1390  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 19:30   
1391  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 19:45   
1392  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 19:50   
1393  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 20:05   
1394  H/42519  Mafi Greif  Stoffer, Andreas  H/42519  17.03.2026 20:10   

               ABFAHRT     GEOX      GEOY  
1318  17.03.2026 06:00  9.98632  53.52348  
1319  17.03.2026 06:05 

## Löschen der Mafi Greif Fahrzeuge (Anweisung von Firma)


In [10]:
df_soll = df_soll[df_soll['LKW_KENNZ'] != 'Mafi Greif']

In [11]:
# Häufigkeit jeder Tournummer
print(df_soll['TOURNR'].value_counts())

TOURNR
H/42510    20
H/42480    17
H/42513    17
H/42488    17
H/42450    17
           ..
H/42570     3
H/42638     3
H/42675     3
H/42696     3
H/42722     3
Name: count, Length: 240, dtype: int64


In [ ]:
# Anzahl aller Zeilen mit einer Tournummer 
print(df_soll['TOURNR'].count())

1991


In [13]:
df_soll

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 06:00,02.03.2026 06:00,9.98632,53.52348
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 06:00,02.03.2026 06:30,9.98632,53.52348
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 06:49,02.03.2026 07:19,9.99383,53.54751
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 07:41,02.03.2026 08:11,9.98632,53.52348
4,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,02.03.2026 08:31,02.03.2026 09:01,9.99383,53.54751
...,...,...,...,...,...,...,...,...
2795,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,07.04.2026 06:00,07.04.2026 06:00,9.98632,53.52348
2796,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,07.04.2026 06:57,07.04.2026 07:27,10.09782,53.53610
2797,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,07.04.2026 07:53,07.04.2026 08:23,9.98632,53.52348
2798,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,07.04.2026 09:52,07.04.2026 10:22,9.51017,53.64711


# Plausibilität zwischen Ankunft und Abfahrt
* Ankunft muss zeitlich vor Abfahrt sein -> alles plausibel
* Untersuchung der Stoppdauer

In [14]:
df_soll['ANKUNFT'] = pd.to_datetime(df_soll['ANKUNFT'], format='%d.%m.%Y %H:%M')
df_soll['ABFAHRT'] = pd.to_datetime(df_soll['ABFAHRT'], format='%d.%m.%Y %H:%M')

# Stoppdauer berechnen
df_soll['STOPP_DAUER'] = df_soll['ABFAHRT'] - df_soll['ANKUNFT']

# Fehlerhafte Einträge: Abfahrt vor Ankunft
fehler_zeit = df_soll[df_soll['STOPP_DAUER'] < pd.Timedelta(0)]
print(f"Zeitfehler: {len(fehler_zeit)} Einträge")
print(fehler_zeit)

Zeitfehler: 0 Einträge
Empty DataFrame
Columns: [TOURNR, LKW_KENNZ, FAHRERNAME, TOUR, ANKUNFT, ABFAHRT, GEOX, GEOY, STOPP_DAUER]
Index: []


In [15]:
df_soll.nunique()

TOURNR          240
LKW_KENNZ        14
FAHRERNAME       15
TOUR            240
ANKUNFT        1516
ABFAHRT        1593
GEOX            237
GEOY            238
STOPP_DAUER      30
dtype: int64

In [16]:
df_soll["FAHRERNAME"].unique()

array(['Hesse, Sven-Erik', 'Szmajdewicz, Marcin', 'Krüger, Alexander',
       'Jorczik, Christian', 'Ledwina, Jens', 'Borrs, Thomas',
       'Calina, Florin Alexandru', 'Oelschlaeger, Marcel',
       'Giewon, Mariusz', 'Staben, Volker', 'Schulz, Frank',
       'Teme, Abdoulaye', 'Hrzuska, Krzystof', 'Sowa, Mariusz', nan,
       'Stoffer, Andreas'], dtype=object)

In [17]:
# Stopps pro Tour sortiert nach Ankunftszeit
df_sorted = df_soll.sort_values(['TOURNR', 'ANKUNFT'])

# Überblick: Wie viele Stopps hat jede Tour?
print(df_sorted.groupby('TOURNR').size().rename('Anzahl Stopps'))

TOURNR
H/42374     6
H/42375     7
H/42376    10
H/42377     6
H/42378     6
           ..
H/42742    16
H/42746     4
H/42750     8
H/42754     7
H/42758     5
Name: Anzahl Stopps, Length: 240, dtype: int64


### Berechnung der Stoppdauer

In [18]:
df_soll['STOPP_DAUER'] = df_soll['ABFAHRT'] - df_soll['ANKUNFT']
fehler_zeit = df_soll[df_soll['STOPP_DAUER'] < pd.Timedelta(0)]
print(f"Zeitfehler: {len(fehler_zeit)} Einträge")

Zeitfehler: 0 Einträge


In [19]:
df_soll['STOPP_DAUER'].describe()

count                         1991
mean     0 days 00:25:48.789552988
std      0 days 00:33:14.678571708
min                0 days 00:00:00
25%                0 days 00:15:00
50%                0 days 00:30:00
75%                0 days 00:30:00
max                0 days 14:08:00
Name: STOPP_DAUER, dtype: object

# Einlesen der Ist-Daten
* 1 Excel pro Fahrzeug
* loop durch alle Dateien, loop durch alle Blätter und dann Formularbeschreibung mit `Ankunft Kunde` und `Abfahrt Kunde` rausgesucht

In [20]:
dateien = [
    'Einzelmeldungsreport_HTS-Cargo_GmbH_1.xlsx',
    'Einzelmeldungsreport_HTS-Cargo_GmbH_2.xlsx',
    'Einzelmeldungsreport_HTS-Cargo_GmbH_3.xlsx',
    'Einzelmeldungsreport_HTS-Cargo_GmbH_4.xlsx',
    'Einzelmeldungsreport_HTS-Cargo_GmbH_5.xlsx',
]

ist_frames = []

for datei in dateien:
    alle_blaetter = pd.read_excel(datei, sheet_name=None, header=8)
    for blatt_name, blatt_df in alle_blaetter.items():
        if 'Formularbeschreibung' in blatt_df.columns:
            gefiltert = blatt_df[blatt_df['Formularbeschreibung'].isin(['Ankunft Kunde', 'Abfahrt Kunde'])].copy()
            gefiltert['Fahrzeug'] = blatt_name  # Fahrzeugname als Spalte speichern
            if not gefiltert.empty:
                ist_frames.append(gefiltert)

df_ist = pd.concat(ist_frames, ignore_index=True)
print(f"Ist-Daten: {len(df_ist)} Einträge")
print(df_ist.columns.tolist())

Ist-Daten: 2875 Einträge
['Fahrer PIN', 'Fahrername', 'Formularnummer', 'Formularbeschreibung', 'Meldungszeit', 'Richtung', 'Geschwindigkeit', 'KM-Stand', 'Gesamtverbrauch', 'Tankfüllstand', 'Breitengrad', 'Längengrad', 'Position', 'Meldungsquelle', 'Fahrzeug']


In [21]:
df_ist = df_ist.drop('Meldungsquelle', axis=1)
df_ist

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 11:49:13,208,0,502860.415,0.0,0.0,"53,51717","9,96982","DE - 20457 Hamburg (Hamburg-Mitte), Köhlbrandb...",TS859 - Fahrzeugdetails
2871,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 11:59:39,0,0,502863.510,0.0,0.0,"53,52261","9,98925","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails
2872,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 12:11:29,0,0,502863.755,0.0,0.0,"53,5228","9,98903","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails
2873,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 12:37:22,0,0,502869.170,0.0,0.0,"53,5341","9,98237","DE - 20457 Hamburg (Hamburg-Mitte), Worthdamm 32",TS859 - Fahrzeugdetails


### Mafi rausfiltern falls enthalten

In [22]:
df_ist = df_ist[~df_ist['Fahrzeug'].str.contains('Mafi', case=False, na=False)]

In [23]:
df_ist

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 11:49:13,208,0,502860.415,0.0,0.0,"53,51717","9,96982","DE - 20457 Hamburg (Hamburg-Mitte), Köhlbrandb...",TS859 - Fahrzeugdetails
2871,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 11:59:39,0,0,502863.510,0.0,0.0,"53,52261","9,98925","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails
2872,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 12:11:29,0,0,502863.755,0.0,0.0,"53,5228","9,98903","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails
2873,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 12:37:22,0,0,502869.170,0.0,0.0,"53,5341","9,98237","DE - 20457 Hamburg (Hamburg-Mitte), Worthdamm 32",TS859 - Fahrzeugdetails


### Datum aus Soll Daten und Ist Daten extrahieren zum Tournummern zuordnen

In [24]:
# Soll-Daten: Datum aus ANKUNFT extrahieren
df_soll['DATUM'] = df_soll['ANKUNFT'].dt.date

# Ist-Daten: Meldungszeit parsen und Datum extrahieren
df_ist['Meldungszeit'] = pd.to_datetime(df_ist['Meldungszeit'], format='%d.%m.%Y %H:%M:%S', errors='coerce')
df_ist['DATUM'] = df_ist['Meldungszeit'].dt.date

In [25]:
df_ist.head()

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails,2026-03-24


In [26]:
df_soll.head()

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
4,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02


## Vergleich von Soll- und Ist-Daten

In [27]:
df_ist.nunique()

Fahrer PIN                11
Fahrername                11
Formularnummer             2
Formularbeschreibung       2
Meldungszeit            2753
Richtung                 314
Geschwindigkeit            5
KM-Stand                1421
Gesamtverbrauch          637
Tankfüllstand            176
Breitengrad             1066
Längengrad              1294
Position                 375
Fahrzeug                  12
DATUM                     24
dtype: int64

In [28]:
df_soll.nunique()

TOURNR          240
LKW_KENNZ        14
FAHRERNAME       15
TOUR            240
ANKUNFT        1516
ABFAHRT        1593
GEOX            237
GEOY            238
STOPP_DAUER      30
DATUM            25
dtype: int64

### Erkenntnisse:
* in Soll-Daten mehr Fahrer und mehr Kennzeichen als in Ist-Daten
* in Soll-Daten ein Datum mehr
* Ist-Daten haben wesentlich mehr Breiten-/Längengrade als Soll-Daten

## Fahrzeugdetails
* jedes Blatt hat im Titel das Kennzeichen

In [29]:
df_ist['Fahrzeug'].unique()

array(['OD-TS 8041 - Fahrzeugdetails', 'OD-TS 8046 - Fahrzeugdetails',
       'OD-TS 8048 - Fahrzeugdetails', 'OD-TS 8050 - Fahrzeugdetails',
       'PI-EN 444 - Fahrzeugdetails', 'PI-EN 900 - Fahrzeugdetails',
       'Plo-TS857 - Fahrzeugdetails', 'Plö-TS853 - Fahrzeugdetails',
       'Plö-TS856 - Fahrzeugdetails', 'TS859 - Fahrzeugdetails',
       'WL-PL431 - Fahrzeugdetails', 'OD-TS 8044 - Fahrzeugdetails'],
      dtype=object)

## Schreibfehler/Unterschiede zwischen den Fahrernamen (bei gleichem Fahrer) -> darum kümmern wir uns nach den Kennzeichen weiter untern

In [30]:
df_ist['Fahrername'].unique()

array(['Jorczik, Christian', 'Calina, Florin Alexandru',
       'Oelschläger, Marcel', 'Szmajdewicz, Marcin', 'Giewon, Mariusz',
       'Teme, Abdoulaeye', 'Ledwina, Jens', 'Borrs, Thomas',
       'Hesse, Sven-Erik', 'Krüger, Alexander', 'Hruszka, Krzysztof'],
      dtype=object)

In [31]:
df_soll['FAHRERNAME'].unique()

array(['Hesse, Sven-Erik', 'Szmajdewicz, Marcin', 'Krüger, Alexander',
       'Jorczik, Christian', 'Ledwina, Jens', 'Borrs, Thomas',
       'Calina, Florin Alexandru', 'Oelschlaeger, Marcel',
       'Giewon, Mariusz', 'Staben, Volker', 'Schulz, Frank',
       'Teme, Abdoulaye', 'Hrzuska, Krzystof', 'Sowa, Mariusz', nan,
       'Stoffer, Andreas'], dtype=object)

In [32]:
df_ist['DATUM'].unique()

array([datetime.date(2026, 3, 24), datetime.date(2026, 3, 25),
       datetime.date(2026, 3, 26), datetime.date(2026, 3, 23),
       datetime.date(2026, 3, 27), datetime.date(2026, 3, 30),
       datetime.date(2026, 3, 31), datetime.date(2026, 4, 1),
       datetime.date(2026, 4, 2), datetime.date(2026, 3, 16),
       datetime.date(2026, 3, 17), datetime.date(2026, 3, 19),
       datetime.date(2026, 3, 18), datetime.date(2026, 3, 20),
       datetime.date(2026, 3, 10), datetime.date(2026, 3, 11),
       datetime.date(2026, 3, 12), datetime.date(2026, 3, 13),
       datetime.date(2026, 3, 9), datetime.date(2026, 3, 2),
       datetime.date(2026, 3, 4), datetime.date(2026, 3, 5),
       datetime.date(2026, 3, 3), datetime.date(2026, 3, 6)], dtype=object)

## Gesamtanzahl Fahrten

In [33]:
# Gesamtanzahl Fahrten (eindeutige Fahrername + Datum Kombinationen)
anzahl_fahrten = df_ist.groupby(['Fahrername', 'DATUM']).ngroups
print(f"Gesamtanzahl Fahrten: {anzahl_fahrten}")

Gesamtanzahl Fahrten: 213


Es gibt Tage mit mehreren Touren. Maximalanzahl der Touren pro Tag ist 3.

In [34]:
df_soll.groupby(['FAHRERNAME', 'DATUM'])['TOURNR'].nunique().max()
# Wenn > 1, gibt es Tage mit mehreren Touren

np.int64(3)

## Eindeutige Touren pro Fahrer und Tag
Wie oft mehr als 1 Tour pro Tag?

In [35]:
# Anzahl eindeutiger Touren pro Fahrer + Tag
touren_pro_tag = df_soll.groupby(['FAHRERNAME', 'DATUM'])['TOURNR'].nunique()

# Wie oft gibt es mehr als 1 Tour pro Tag
print((touren_pro_tag > 1).sum(), "Fahrer-Tag-Kombinationen mit mehr als 1 Tour")
print(touren_pro_tag.value_counts().sort_index())

9 Fahrer-Tag-Kombinationen mit mehr als 1 Tour
TOURNR
1    220
2      7
3      2
Name: count, dtype: int64


## Umgang mit Tagen, die mehr als eine Tour haben

In [36]:
# Tage mit mehr als 1 Tour identifizieren
touren_pro_tag = df_soll.groupby(['FAHRERNAME', 'DATUM'])['TOURNR'].nunique()
mehrfach_mask = touren_pro_tag[touren_pro_tag > 1].reset_index()
mehrfach_mask.columns = ['FAHRERNAME', 'DATUM', 'ANZAHL_TOUREN']

# Tournummern und Kennzeichen dazu joinen
details = df_soll.groupby(['FAHRERNAME', 'DATUM']).agg(
    TOURNUMMERN=('TOURNR', lambda x: ', '.join(x.unique())),
    KENNZEICHEN=('LKW_KENNZ', lambda x: ', '.join(x.unique()))
).reset_index()

# Zusammenführen
ergebnis = mehrfach_mask.merge(details, on=['FAHRERNAME', 'DATUM']).sort_values(['ANZAHL_TOUREN', 'DATUM'], ascending=[False, True])

print(ergebnis.to_string(index=False))

              FAHRERNAME      DATUM  ANZAHL_TOUREN               TOURNUMMERN                    KENNZEICHEN
     Szmajdewicz, Marcin 2026-03-19              3 H/42547, H/42549, H/42556 PI-EN 900, OD-TS8050, WL-PL431
     Szmajdewicz, Marcin 2026-03-20              3 H/42561, H/42568, H/42587 OD-TS8050, WL-PL431, PI-EN 900
       Krüger, Alexander 2026-03-03              2          H/42393, H/42415           Plo-TS859, PI-EN 900
         Teme, Abdoulaye 2026-03-11              2          H/42468, H/42505           PI-EN 900, OD-TS8050
     Szmajdewicz, Marcin 2026-03-18              2          H/42536, H/42544            OD-TS8050, WL-PL431
Calina, Florin Alexandru 2026-03-23              2          H/42571, H/42578           PI-EN 900, OD-TS8046
Calina, Florin Alexandru 2026-03-26              2          H/42601, H/42628            OD-TS8046, WL-PL431
       Krüger, Alexander 2026-03-31              2          H/42646, H/42657                      Plo-TS859
    Oelschlaeger, Marcel 202

### Erkenntnisse:
* manche Fahrer haben für denselben Tag mehrere Tournummern und mehrere Kennzeichen, 
Überlegung ist nun so die Tournummern zuzuordnen wenn Kennzeichen, Datum und Person übereinstimmen. 
Für diesen Fall gibt es nur eine Ausnahme, siehe Krüger Alexander am 31.3 ist zwei Touren mit dem selben Fahrzeug gefahren.

Deshalb werden wir folgt die Kennzeichen von df_ist mit denen von df_soll anpassen (es gibt wieder Probleme mit umlauten zb.)

In [37]:
df_soll

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
4,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
...,...,...,...,...,...,...,...,...,...,...
2795,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:00:00,2026-04-07 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-04-07
2796,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:57:00,2026-04-07 07:27:00,10.09782,53.53610,0 days 00:30:00,2026-04-07
2797,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 07:53:00,2026-04-07 08:23:00,9.98632,53.52348,0 days 00:30:00,2026-04-07
2798,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 09:52:00,2026-04-07 10:22:00,9.51017,53.64711,0 days 00:30:00,2026-04-07


### Blatttitel beinhaltet auch das Kennzeichen des Fahrzeugs -> Extrahieren des Kennzeichens

In [38]:
df_ist["Fahrzeug"].unique()

array(['OD-TS 8041 - Fahrzeugdetails', 'OD-TS 8046 - Fahrzeugdetails',
       'OD-TS 8048 - Fahrzeugdetails', 'OD-TS 8050 - Fahrzeugdetails',
       'PI-EN 444 - Fahrzeugdetails', 'PI-EN 900 - Fahrzeugdetails',
       'Plo-TS857 - Fahrzeugdetails', 'Plö-TS853 - Fahrzeugdetails',
       'Plö-TS856 - Fahrzeugdetails', 'TS859 - Fahrzeugdetails',
       'WL-PL431 - Fahrzeugdetails', 'OD-TS 8044 - Fahrzeugdetails'],
      dtype=object)

In [39]:
df_soll["LKW_KENNZ"].unique()

array(['Plo-TS856', 'OD-TS8050', 'Plo-TS859', 'OD-TS8041', 'OD-TS8044',
       'Plö-TS853', 'OD-TS8046', 'OD-TS8048', 'PI-EN 444', 'RW435',
       'PI-EN 900', 'WL-PL431', 'Leihwagen', 'Plö-TS857'], dtype=object)

In [ ]:
# Endung " - Fahrzeugdetails" entfernen
df_ist['KENNZ_CLEAN'] = df_ist['Fahrzeug'].str.replace(' - Fahrzeugdetails', '', regex=False)

# Leerzeichen hinter OD-TS entfernen
df_ist['KENNZ_CLEAN'] = df_ist['KENNZ_CLEAN'].str.replace('OD-TS ', 'OD-TS', regex=False)

# Manuelle Korrekturen
korrekturen = {
    'Plo-TS857': 'Plö-TS857',
    'Plö-TS856': 'Plo-TS856',
    'TS859':     'Plo-TS859'
}
df_ist['KENNZ_CLEAN'] = df_ist['KENNZ_CLEAN'].replace(korrekturen)

In [41]:
df_soll["LKW_KENNZ"].unique()

array(['Plo-TS856', 'OD-TS8050', 'Plo-TS859', 'OD-TS8041', 'OD-TS8044',
       'Plö-TS853', 'OD-TS8046', 'OD-TS8048', 'PI-EN 444', 'RW435',
       'PI-EN 900', 'WL-PL431', 'Leihwagen', 'Plö-TS857'], dtype=object)

In [42]:
df_ist['KENNZ_CLEAN'].unique()

array(['OD-TS8041', 'OD-TS8046', 'OD-TS8048', 'OD-TS8050', 'PI-EN 444',
       'PI-EN 900', 'Plö-TS857', 'Plö-TS853', 'Plo-TS856', 'Plo-TS859',
       'WL-PL431', 'OD-TS8044'], dtype=object)

Kennzeichen von Soll- und Ist-Daten sind nun gleich

## Untersuchung und Korrektur der Fahrernamen bei Umstimmigkeiten zwischen Soll- und Ist-Daten
Am Ende sollen die Fahrernamen gleich sein.

In [43]:
df_soll["FAHRERNAME"].unique()

array(['Hesse, Sven-Erik', 'Szmajdewicz, Marcin', 'Krüger, Alexander',
       'Jorczik, Christian', 'Ledwina, Jens', 'Borrs, Thomas',
       'Calina, Florin Alexandru', 'Oelschlaeger, Marcel',
       'Giewon, Mariusz', 'Staben, Volker', 'Schulz, Frank',
       'Teme, Abdoulaye', 'Hrzuska, Krzystof', 'Sowa, Mariusz', nan,
       'Stoffer, Andreas'], dtype=object)

In [44]:
df_ist['Fahrername'].unique()

array(['Jorczik, Christian', 'Calina, Florin Alexandru',
       'Oelschläger, Marcel', 'Szmajdewicz, Marcin', 'Giewon, Mariusz',
       'Teme, Abdoulaeye', 'Ledwina, Jens', 'Borrs, Thomas',
       'Hesse, Sven-Erik', 'Krüger, Alexander', 'Hruszka, Krzysztof'],
      dtype=object)

In [ ]:
namen_korrekturen = {
    'Oelschläger, Marcel': 'Oelschlaeger, Marcel',
    'Teme, Abdoulaeye':    'Teme, Abdoulaye',
    'Hruszka, Krzysztof':  'Hrzuska, Krzystof'  # df_soll als Referenz
}

df_ist['Fahrername'] = df_ist['Fahrername'].replace(namen_korrekturen)

print(set(df_ist['Fahrername'].unique()) - set(df_soll['FAHRERNAME'].unique()))


set()


In [46]:
print(set(df_ist['KENNZ_CLEAN'].unique()) - set(df_soll['LKW_KENNZ'].unique()))

set()


Hier schauen welche Touren sich nicht eindeutig durch Name, Kennzeichen und Datum unterscheiden.  

In [47]:
touren_detail = df_soll.groupby(['FAHRERNAME', 'LKW_KENNZ', 'DATUM'])['TOURNR'].nunique()

# Nur die mit mehr als 1 Tour
mehrfach = touren_detail[touren_detail > 1].reset_index()
mehrfach.columns = ['FAHRERNAME', 'LKW_KENNZ', 'DATUM', 'ANZAHL_TOUREN']

In [48]:
# Tournummern dazu
tournr = df_soll.groupby(['FAHRERNAME', 'LKW_KENNZ', 'DATUM'])['TOURNR'].apply(lambda x: ', '.join(x.unique())).reset_index()
tournr.columns = ['FAHRERNAME', 'LKW_KENNZ', 'DATUM', 'TOURNUMMERN']

ergebnis = mehrfach.merge(tournr, on=['FAHRERNAME', 'LKW_KENNZ', 'DATUM']).sort_values(['ANZAHL_TOUREN', 'DATUM'], ascending=[False, True])

print(f"Anzahl betroffener Fahrer-Fahrzeug-Tag Kombinationen: {len(ergebnis)}")
print(ergebnis.to_string(index=False))

Anzahl betroffener Fahrer-Fahrzeug-Tag Kombinationen: 1
       FAHRERNAME LKW_KENNZ      DATUM  ANZAHL_TOUREN      TOURNUMMERN
Krüger, Alexander Plo-TS859 2026-03-31              2 H/42646, H/42657


In [ ]:
# Mapping erstellen (eine Tournr pro Fahrer+Kennz+Datum)
tour_mapping = df_soll.groupby(['FAHRERNAME', 'LKW_KENNZ', 'DATUM'])['TOURNR'].apply(lambda x: x.iloc[0]).reset_index()  # erstmal erste Tour nehmen
tour_mapping.columns = ['Fahrername', 'KENNZ_CLEAN', 'DATUM', 'TOURNR']

#  Merge auf df_ist
df_ist_merged = df_ist.merge(tour_mapping, on=['Fahrername', 'KENNZ_CLEAN', 'DATUM'], how='left')

print(df_ist_merged['TOURNR'].isna().sum(), "Zeilen ohne Tournummer")
print(df_ist_merged['TOURNR'].notna().sum(), "Zeilen mit Tournummer")

261 Zeilen ohne Tournummer
2614 Zeilen mit Tournummer


Im Fall Krüger wurde hier nur H/42646 zugewiesen. Da es nur der Einzelfall ist schien das einfacher!
Damit können wir uns erstmal um die groben Fälle kümmern und diese untersuchen.

Nun wurden die Tournummern zu df_ist hinzugefügt, das hat größtenteils geklappt, es gibt circa 10 Prozent von Fällen, wo es nicht geklappt hat, das untersuchen wir weiter unten. 

### Untersuchung welche touren nicht zugeordnet werden konnten

In [50]:
df_ist_merged[df_ist_merged.TOURNR.isna()]

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
253,8314.0,"Teme, Abdoulaye",1650,Ankunft Kunde,2026-03-23 11:45:25,0,0,143826.335,45835.272,44.0,"53,51678","9,96972","DE - 20457 Hamburg (Hamburg-Mitte), Neuhöfer D...",PI-EN 900 - Fahrzeugdetails,2026-03-23,PI-EN 900,NaN
254,8314.0,"Teme, Abdoulaye",1651,Abfahrt Kunde,2026-03-23 11:45:26,0,0,143826.335,45835.272,44.0,"53,51678","9,96972","DE - 20457 Hamburg (Hamburg-Mitte), Neuhöfer D...",PI-EN 900 - Fahrzeugdetails,2026-03-23,PI-EN 900,NaN
255,8314.0,"Teme, Abdoulaye",1650,Ankunft Kunde,2026-03-23 14:09:15,0,0,143909.580,45861.824,24.0,"53,63088","9,50845","DE - 21683 Stade, Stader Elbstraße",PI-EN 900 - Fahrzeugdetails,2026-03-23,PI-EN 900,NaN
256,8314.0,"Teme, Abdoulaye",1651,Abfahrt Kunde,2026-03-23 14:09:16,0,0,143909.580,45861.824,24.0,"53,63088","9,50845","DE - 21683 Stade, Stader Elbstraße",PI-EN 900 - Fahrzeugdetails,2026-03-23,PI-EN 900,NaN
480,8311.0,"Szmajdewicz, Marcin",1650,Ankunft Kunde,2026-03-24 07:48:59,329,0,807302.125,0.000,0.0,"53,52322","9,98857","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",WL-PL431 - Fahrzeugdetails,2026-03-24,WL-PL431,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2702,8197.0,"Hrzuska, Krzystof",1651,Abfahrt Kunde,2026-03-06 13:50:54,0,0,179191.380,50937.052,32.0,"53,5226","9,98912","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",PI-EN 444 - Fahrzeugdetails,2026-03-06,PI-EN 444,NaN
2703,8197.0,"Hrzuska, Krzystof",1650,Ankunft Kunde,2026-03-06 13:50:57,0,0,179191.380,50937.052,32.0,"53,5226","9,98912","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",PI-EN 444 - Fahrzeugdetails,2026-03-06,PI-EN 444,NaN
2704,8197.0,"Hrzuska, Krzystof",1651,Abfahrt Kunde,2026-03-06 13:50:59,0,0,179191.380,50937.052,32.0,"53,5226","9,98912","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",PI-EN 444 - Fahrzeugdetails,2026-03-06,PI-EN 444,NaN
2705,8317.0,"Hesse, Sven-Erik",1650,Ankunft Kunde,2026-03-03 06:55:23,0,0,143469.000,45685.036,4.8,"53,53563","9,97776","DE - 20457 Hamburg (Hamburg-Mitte), Stillhorne...",PI-EN 900 - Fahrzeugdetails,2026-03-03,PI-EN 900,NaN


In [51]:
# Welche Fahrer-Kennz-Datum Kombinationen haben kein Match?
keine_tour = df_ist_merged[df_ist_merged['TOURNR'].isna()]

print("Fehlende Matches nach Fahrer")
print(keine_tour.groupby('Fahrername').size().sort_values(ascending=False))

print("\nFehlende Matches nach KENNZ_CLEAN")
print(keine_tour.groupby('KENNZ_CLEAN').size().sort_values(ascending=False))

print("\nFehlende Matches nach DATUM")
print(keine_tour.groupby('DATUM').size().sort_values(ascending=False))

print("\nFahrer+Kennz+Datum ohne Match")
print(keine_tour.groupby(['Fahrername', 'KENNZ_CLEAN', 'DATUM']).size().reset_index().to_string(index=False))

Fehlende Matches nach Fahrer
Fahrername
Hrzuska, Krzystof           224
Teme, Abdoulaye              22
Szmajdewicz, Marcin           8
Calina, Florin Alexandru      4
Hesse, Sven-Erik              2
Ledwina, Jens                 1
dtype: int64

Fehlende Matches nach KENNZ_CLEAN
KENNZ_CLEAN
PI-EN 444    138
OD-TS8050     70
WL-PL431      35
PI-EN 900     17
OD-TS8044      1
dtype: int64

Fehlende Matches nach DATUM
DATUM
2026-03-19    37
2026-03-05    33
2026-03-06    29
2026-03-02    28
2026-03-03    26
2026-03-04    24
2026-03-20    21
2026-03-18    20
2026-03-13    20
2026-03-24     8
2026-03-09     4
2026-03-23     4
2026-04-01     4
2026-03-26     3
dtype: int64

Fahrer+Kennz+Datum ohne Match
              Fahrername KENNZ_CLEAN      DATUM  0
Calina, Florin Alexandru   OD-TS8050 2026-03-09  4
        Hesse, Sven-Erik   PI-EN 900 2026-03-03  2
       Hrzuska, Krzystof   OD-TS8050 2026-03-18 20
       Hrzuska, Krzystof   OD-TS8050 2026-03-19 26
       Hrzuska, Krzystof   OD-TS8050 2

Sind die Beiden Fahrer Ersatzfahrer für Krankheitsfälle?

In [ ]:
# 1. Welche Kennzeichen fährt Hrzuska in df_ist?
print("Hrzuska in df_ist")
print(df_ist[df_ist['Fahrername'] == 'Hrzuska, Krzystof'].groupby(['KENNZ_CLEAN', 'DATUM']).size().reset_index().to_string(index=False))

# 2. Welche Kennzeichen hat Hrzuska in df_soll?
print("\nHrzuska in df_soll")
print(df_soll[df_soll['FAHRERNAME'] == 'Hrzuska, Krzystof'].groupby(['LKW_KENNZ', 'DATUM']).size().reset_index().to_string(index=False))

# 3. Für Teme, welche Kennzeichen fehlen?
print("\nTeme in df_ist")
keine_tour = df_ist_merged[df_ist_merged['TOURNR'].isna()]
print(keine_tour[keine_tour['Fahrername'] == 'Teme, Abdoulaye'].groupby(['KENNZ_CLEAN', 'DATUM']).size().reset_index().to_string(index=False))

print("\nTeme in df_soll")
print(df_soll[df_soll['FAHRERNAME'] == 'Teme, Abdoulaye'].groupby(['LKW_KENNZ', 'DATUM']).size().reset_index().to_string(index=False))

Hrzuska in df_ist
KENNZ_CLEAN      DATUM  0
  OD-TS8050 2026-03-17 25
  OD-TS8050 2026-03-18 20
  OD-TS8050 2026-03-19 26
  OD-TS8050 2026-03-20 20
  PI-EN 444 2026-03-02 28
  PI-EN 444 2026-03-03 24
  PI-EN 444 2026-03-04 24
  PI-EN 444 2026-03-05 33
  PI-EN 444 2026-03-06 29
  PI-EN 900 2026-03-16 22
   WL-PL431 2026-03-09 36
   WL-PL431 2026-03-10 20
   WL-PL431 2026-03-11 34
   WL-PL431 2026-03-12 16
   WL-PL431 2026-03-13 20

Hrzuska in df_soll
LKW_KENNZ      DATUM  0
OD-TS8050 2026-03-17 15
PI-EN 900 2026-03-16 13
 WL-PL431 2026-03-09 17
 WL-PL431 2026-03-10 13
 WL-PL431 2026-03-11 13
 WL-PL431 2026-03-12 12

Teme in df_ist
KENNZ_CLEAN      DATUM  0
  PI-EN 900 2026-03-19 11
  PI-EN 900 2026-03-23  4
   WL-PL431 2026-03-26  3
   WL-PL431 2026-04-01  4

Teme in df_soll
LKW_KENNZ      DATUM  0
OD-TS8050 2026-03-11  3
OD-TS8050 2026-03-12  4
OD-TS8050 2026-03-13  6
PI-EN 444 2026-04-07  5
PI-EN 900 2026-03-09 10
PI-EN 900 2026-03-10  7
PI-EN 900 2026-03-11  8
PI-EN 900 2026-03-17  9

In [53]:
df_ist_merged

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 11:49:13,208,0,502860.415,0.0,0.0,"53,51717","9,96982","DE - 20457 Hamburg (Hamburg-Mitte), Köhlbrandb...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2871,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 11:59:39,0,0,502863.510,0.0,0.0,"53,52261","9,98925","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2872,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 12:11:29,0,0,502863.755,0.0,0.0,"53,5228","9,98903","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2873,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 12:37:22,0,0,502869.170,0.0,0.0,"53,5341","9,98237","DE - 20457 Hamburg (Hamburg-Mitte), Worthdamm 32",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432


In [ ]:
# Finale Statistik
gesamt = len(df_ist_merged)
mit_tour = df_ist_merged['TOURNR'].notna().sum()
ohne_tour = df_ist_merged['TOURNR'].isna().sum()

print(f"Gesamt Meldungen:       {gesamt}")
print(f"Mit Tournummer:         {mit_tour} ({mit_tour/gesamt*100:.1f}%)")
print(f"Ohne Tournummer (NaN):  {ohne_tour} ({ohne_tour/gesamt*100:.1f}%)")

df_luecken = df_ist_merged[df_ist_merged['TOURNR'].isna()].groupby(['Fahrername', 'KENNZ_CLEAN', 'DATUM']).size().reset_index()
df_luecken.columns = ['Fahrername', 'Kennzeichen', 'Datum', 'Anzahl_Meldungen']
df_luecken['Grund'] = 'Kein Soll-Eintrag vorhanden'
print(df_luecken)

Gesamt Meldungen:       2875
Mit Tournummer:         2614 (90.9%)
Ohne Tournummer (NaN):  261 (9.1%)
                  Fahrername Kennzeichen       Datum  Anzahl_Meldungen  \
0   Calina, Florin Alexandru   OD-TS8050  2026-03-09                 4   
1           Hesse, Sven-Erik   PI-EN 900  2026-03-03                 2   
2          Hrzuska, Krzystof   OD-TS8050  2026-03-18                20   
3          Hrzuska, Krzystof   OD-TS8050  2026-03-19                26   
4          Hrzuska, Krzystof   OD-TS8050  2026-03-20                20   
5          Hrzuska, Krzystof   PI-EN 444  2026-03-02                28   
6          Hrzuska, Krzystof   PI-EN 444  2026-03-03                24   
7          Hrzuska, Krzystof   PI-EN 444  2026-03-04                24   
8          Hrzuska, Krzystof   PI-EN 444  2026-03-05                33   
9          Hrzuska, Krzystof   PI-EN 444  2026-03-06                29   
10         Hrzuska, Krzystof    WL-PL431  2026-03-13                20   
11         

In [55]:
df_ist_merged["TOURNR"].nunique()

200

### Untersuchung ob Touren in df_soll sind aber nicht in df_ist

Es sind ca 40 Touren in df_soll aber nicht in df_ist. 
Im Kontrast dazu sind 14 Touren in df_ist aber nicht df_soll. 
Das bezieht sich jetzt auf das Mapping Datum Fahrer und Kennzeichen. 
Natürlich könnten einige der Fahrten die nicht in df_soll sind, Ersatzfahrten sein, weil ein Fahrer krank wurde.
Allerdings denken wir, fürs erste ist es sinnvoll mit den Daten weiterzuarbeiten, die wir Tournummern-mäßig zuordnen konnten. Das sind 200. 
Mit diesen können wir das Geofencing ausprobeiren und starten. 

In [56]:
df_soll["TOURNR"].nunique()

240

In [57]:
# Welche Touren sind in df_soll aber nicht in df_ist?
touren_soll = set(df_soll['TOURNR'].unique())
touren_ist = set(df_ist_merged['TOURNR'].dropna().unique())

fehlende_touren = touren_soll - touren_ist

# Details zu den fehlenden Touren
df_fehlend = df_soll[df_soll['TOURNR'].isin(fehlende_touren)][['TOURNR', 'FAHRERNAME', 'LKW_KENNZ', 'DATUM']].drop_duplicates().sort_values('DATUM')

print(f"Fehlende Touren: {len(fehlende_touren)}")
print(df_fehlend.to_string(index=False))

Fehlende Touren: 40
 TOURNR               FAHRERNAME LKW_KENNZ      DATUM
H/42383          Giewon, Mariusz PI-EN 444 2026-03-02
H/42416            Schulz, Frank PI-EN 900 2026-03-02
H/42400          Giewon, Mariusz PI-EN 444 2026-03-03
H/42415        Krüger, Alexander PI-EN 900 2026-03-03
H/42412          Giewon, Mariusz PI-EN 444 2026-03-04
H/42413           Staben, Volker     RW435 2026-03-04
H/42427          Giewon, Mariusz PI-EN 444 2026-03-05
H/42439          Giewon, Mariusz PI-EN 444 2026-03-06
H/42429            Schulz, Frank PI-EN 900 2026-03-06
H/42447            Schulz, Frank OD-TS8050 2026-03-09
H/42449       Jorczik, Christian OD-TS8041 2026-03-09
H/42452 Calina, Florin Alexandru OD-TS8046 2026-03-09
H/42467           Staben, Volker     RW435 2026-03-10
H/42493           Staben, Volker     RW435 2026-03-12
H/42503            Sowa, Mariusz  WL-PL431 2026-03-13
H/42533           Staben, Volker     RW435 2026-03-17
H/42536      Szmajdewicz, Marcin OD-TS8050 2026-03-18
H/42570 

Sowa, Mariusz & Stoffer, Andreas, Frank Schulz und Staben Volker tauchen in df_ist nie auf
04.07. war massiver Ausfall, war in einzelmeldungsreport tatsächlich gar nicht inkludiert -> gar nicht in ist daten
Staben, Volker fährt immer (RW435)  Kennzeichen taucht in df_ist nie auf

-> ich würde nur mit den Zugeordneten touren weiter arbeiten

In [ ]:
# Wer ist gar nicht in df_ist?
print("In df_soll aber nie in df_ist:")
print(set(df_soll['FAHRERNAME'].dropna()) - set(df_ist['Fahrername'].unique()))

# Wie viele fehlende Touren pro Fahrer?
print("\nFehlende Touren pro Fahrer:")
print(df_fehlend.groupby('FAHRERNAME').size().sort_values(ascending=False))

# Wie viele fehlende Touren pro Datum?
print("\nFehlende Touren pro Datum:")
print(df_fehlend.groupby('DATUM').size().sort_values(ascending=False))

In df_soll aber nie in df_ist:
{'Sowa, Mariusz', 'Stoffer, Andreas', 'Staben, Volker', 'Schulz, Frank'}

Fehlende Touren pro Fahrer:
FAHRERNAME
Staben, Volker              7
Giewon, Mariusz             5
Schulz, Frank               4
Szmajdewicz, Marcin         4
Krüger, Alexander           3
Jorczik, Christian          3
Teme, Abdoulaye             3
Oelschlaeger, Marcel        2
Calina, Florin Alexandru    2
Sowa, Mariusz               2
Hesse, Sven-Erik            1
Borrs, Thomas               1
Ledwina, Jens               1
Stoffer, Andreas            1
dtype: int64

Fehlende Touren pro Datum:
DATUM
2026-04-07    10
2026-03-19     4
2026-03-09     3
2026-03-02     2
2026-03-20     2
2026-03-04     2
2026-03-03     2
2026-03-06     2
2026-03-25     2
2026-03-05     1
2026-03-12     1
2026-03-10     1
2026-03-18     1
2026-03-17     1
2026-03-13     1
2026-03-23     1
2026-03-24     1
2026-03-31     1
2026-04-01     1
2026-04-02     1
dtype: int64


-->> Nur mit zugeorndeten Tournummern weiter arbeiten !

In [59]:
df_ist_merged


,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 11:49:13,208,0,502860.415,0.0,0.0,"53,51717","9,96982","DE - 20457 Hamburg (Hamburg-Mitte), Köhlbrandb...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2871,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 11:59:39,0,0,502863.510,0.0,0.0,"53,52261","9,98925","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2872,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 12:11:29,0,0,502863.755,0.0,0.0,"53,5228","9,98903","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2873,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 12:37:22,0,0,502869.170,0.0,0.0,"53,5341","9,98237","DE - 20457 Hamburg (Hamburg-Mitte), Worthdamm 32",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432


### Arbeit an Übersichtlichkeit von df_soll


Hier ist auffällig:
1. Es gibt bei identischer Tournummer identiscche GEOX, GEOY und Ankunft aber verschiedene Abfahrten.
Uns scheint die späteren Abfahrten machen mehr Sinn -> wir filtern bei Identischer Tournummer, GEOX, GEOY und Ankunft die stopps raus, die frühere Abfahrt haben.

2. In den Ist Daten stoppt der Plan nach der "letzten Warenausgabe". Bei den Soll daten stoppt der Plan nach Rückankunft beim Ausgangsort.
Deshalb in Fällen wo bei einer Tour der erste und letzte Stopp identisch sind Filtern wir den letzten Stopp sodass wir dann besser mir Ist Daten vergleichen können.

In [60]:
df_soll

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
4,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
...,...,...,...,...,...,...,...,...,...,...
2795,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:00:00,2026-04-07 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-04-07
2796,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:57:00,2026-04-07 07:27:00,10.09782,53.53610,0 days 00:30:00,2026-04-07
2797,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 07:53:00,2026-04-07 08:23:00,9.98632,53.52348,0 days 00:30:00,2026-04-07
2798,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 09:52:00,2026-04-07 10:22:00,9.51017,53.64711,0 days 00:30:00,2026-04-07


Schauen wie oft tatsächlich zwei identische "Starts" sind:

In [61]:
dupe_groups = df_soll.groupby(['TOURNR', 'GEOX', 'GEOY', 'ANKUNFT']).size()
print(dupe_groups[dupe_groups > 1].sort_values(ascending=False))

TOURNR   GEOX     GEOY      ANKUNFT            
H/42374  9.98632  53.52348  2026-03-02 06:00:00    2
H/42376  9.98632  53.52348  2026-03-02 06:00:00    2
H/42379  9.99189  53.66737  2026-03-02 05:00:00    2
H/42380  9.98632  53.52348  2026-03-02 06:00:00    2
H/42383  9.98632  53.52348  2026-03-02 06:00:00    2
                                                  ..
H/42708  9.98632  53.52348  2026-04-01 06:00:00    2
H/42718  9.98632  53.52348  2026-04-07 06:00:00    2
H/42738  9.99189  53.66737  2026-04-07 05:00:00    2
H/42742  9.99189  53.66737  2026-04-07 05:00:00    2
H/42746  9.98632  53.52348  2026-04-07 06:00:00    2
Length: 135, dtype: int64


In [ ]:
# Sortiere nach abfahrt
df_soll_sorted = df_soll.sort_values(['TOURNR', 'GEOX', 'GEOY', 'ANKUNFT', 'ABFAHRT'], ascending=[True, True, True, True, False])

# Behalte nur die späteste Abfahrt
df_filtered = df_soll_sorted.drop_duplicates(subset=['TOURNR', 'GEOX', 'GEOY', 'ANKUNFT'], keep='first')

print(f"Anzahl Duplikate entfernt: {len(df_soll) - len(df_filtered)}")

Anzahl Duplikate entfernt: 135


Hier sieht man dass das entfernen der Duplikate geklappt hat

In [63]:
df_filtered

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
5,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 09:23:00,2026-03-02 09:23:00,9.98632,53.52348,0 days 00:00:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
4,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
...,...,...,...,...,...,...,...,...,...,...
2798,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 09:52:00,2026-04-07 10:22:00,9.51017,53.64711,0 days 00:30:00,2026-04-07
2795,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:00:00,2026-04-07 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-04-07
2797,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 07:53:00,2026-04-07 08:23:00,9.98632,53.52348,0 days 00:30:00,2026-04-07
2799,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 11:52:00,2026-04-07 11:52:00,9.98632,53.52348,0 days 00:00:00,2026-04-07


In [64]:
df_filtered[df_filtered['TOURNR'] == "H/42380"]

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
47,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 07:59:00,2026-03-02 08:29:00,9.51017,53.64711,0 days 00:30:00,2026-03-02
46,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
48,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 09:59:00,2026-03-02 10:29:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
50,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 12:42:00,2026-03-02 12:42:00,9.98632,53.52348,0 days 00:00:00,2026-03-02
49,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 11:19:00,2026-03-02 11:49:00,9.99022,53.72624,0 days 00:30:00,2026-03-02


Jetzt ist allerdings alles nach Tournummer nach GEOX geordnet, also machen wir das mal zurück

In [65]:
df_final = df_filtered.sort_values(['TOURNR', 'DATUM', 'ABFAHRT']).reset_index(drop=True)

In [66]:
df_final[df_final['TOURNR'] == "H/42380"]

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
42,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
43,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 07:59:00,2026-03-02 08:29:00,9.51017,53.64711,0 days 00:30:00,2026-03-02
44,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 09:59:00,2026-03-02 10:29:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
45,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 11:19:00,2026-03-02 11:49:00,9.99022,53.72624,0 days 00:30:00,2026-03-02
46,H/42380,OD-TS8046,"Calina, Florin Alexandru",H/42380,2026-03-02 12:42:00,2026-03-02 12:42:00,9.98632,53.52348,0 days 00:00:00,2026-03-02


Jetzt sind touren wieder zeitmäßig sortiert. 
Nun schauen wir wann die Anfangs- und stopppunkte wirklich identisch sind.

In [ ]:
# Pro Tour die Koordinaten vom ersten und letzten Stopp holen
tour_starts = df_final.groupby('TOURNR').first()[['GEOX', 'GEOY']]
tour_ends = df_final.groupby('TOURNR').last()[['GEOX', 'GEOY']]

# Vergleichen ob Start gleich Ende
tour_starts.columns = ['start_x', 'start_y']
tour_ends.columns = ['end_x', 'end_y']
comparison = tour_starts.join(tour_ends)

matches = ((comparison['start_x'] == comparison['end_x']) & (comparison['start_y'] == comparison['end_y'])).sum()

print(f"{matches} Touren starten und enden am selben Ort")

236 Touren starten und enden am selben Ort


In [68]:
nicht_identisch = comparison[
    (comparison['start_x'] != comparison['end_x']) | 
    (comparison['start_y'] != comparison['end_y'])
]
touren_ohne_kreis = nicht_identisch.index.tolist()
print("Touren ohne Kreis:", touren_ohne_kreis)

Touren ohne Kreis: ['H/42459', 'H/42468', 'H/42510', 'H/42633']


Ausprobieren der Touren die nicht identischen Start und Stop haben.

In [69]:
df_final[df_final['TOURNR'] == "H/42459"]

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
434,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 06:00:00,2026-03-10 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-10
435,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 06:30:00,2026-03-10 07:00:00,9.99383,53.54751,0 days 00:30:00,2026-03-10
436,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 07:00:00,2026-03-10 07:30:00,0.00000,0.00000,0 days 00:30:00,2026-03-10
437,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 07:30:00,2026-03-10 08:00:00,9.98632,53.52348,0 days 00:30:00,2026-03-10
438,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 08:00:00,2026-03-10 08:30:00,9.98213,53.53497,0 days 00:30:00,2026-03-10
439,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 08:30:00,2026-03-10 09:00:00,0.00000,0.00000,0 days 00:30:00,2026-03-10
440,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 09:00:00,2026-03-10 09:30:00,9.98213,53.53497,0 days 00:30:00,2026-03-10
441,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 09:30:00,2026-03-10 10:00:00,9.98632,53.52348,0 days 00:30:00,2026-03-10
442,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 10:00:00,2026-03-10 10:30:00,0.00000,0.00000,0 days 00:30:00,2026-03-10
443,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 10:30:00,2026-03-10 11:00:00,9.98632,53.52348,0 days 00:30:00,2026-03-10


### Koordinaten fehlen
Bei 10 Zeilen (2 Touren fehlen die Koordinaten) -> Was tun? Zeilen entfernen? Touren entfernen? Adresse raussuchen?

In [70]:
df_final[(df_final['GEOX'] == 0) | (df_final.GEOY == 0)]

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
436,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 07:00:00,2026-03-10 07:30:00,0.0,0.0,0 days 00:30:00,2026-03-10
439,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 08:30:00,2026-03-10 09:00:00,0.0,0.0,0 days 00:30:00,2026-03-10
442,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 10:00:00,2026-03-10 10:30:00,0.0,0.0,0 days 00:30:00,2026-03-10
444,H/42459,Plo-TS859,"Krüger, Alexander",H/42459,2026-03-10 11:00:00,2026-03-10 11:30:00,0.0,0.0,0 days 00:30:00,2026-03-10
805,H/42510,Plo-TS859,"Krüger, Alexander",H/42510,2026-03-16 07:00:00,2026-03-16 07:30:00,0.0,0.0,0 days 00:30:00,2026-03-16
807,H/42510,Plo-TS859,"Krüger, Alexander",H/42510,2026-03-16 08:00:00,2026-03-16 08:30:00,0.0,0.0,0 days 00:30:00,2026-03-16
810,H/42510,Plo-TS859,"Krüger, Alexander",H/42510,2026-03-16 09:30:00,2026-03-16 10:00:00,0.0,0.0,0 days 00:30:00,2026-03-16
812,H/42510,Plo-TS859,"Krüger, Alexander",H/42510,2026-03-16 10:30:00,2026-03-16 11:00:00,0.0,0.0,0 days 00:30:00,2026-03-16
816,H/42510,Plo-TS859,"Krüger, Alexander",H/42510,2026-03-16 12:30:00,2026-03-16 13:00:00,0.0,0.0,0 days 00:30:00,2026-03-16
818,H/42510,Plo-TS859,"Krüger, Alexander",H/42510,2026-03-16 13:30:00,2026-03-16 14:00:00,0.0,0.0,0 days 00:30:00,2026-03-16


Dennoch in den 236 Fällen wo Start und Endpunkt identisch sind werde ich den Endpunkt rausfiltern, sodass wir dann Starten können die Fälle zu vergleichen.

In [71]:
kreis_touren = comparison[
    (comparison['start_x'] == comparison['end_x']) & 
    (comparison['start_y'] == comparison['end_y'])
].index

df_clean = df_final.copy()
for tour in kreis_touren:
    tour_data = df_clean[df_clean['TOURNR'] == tour]
    if len(tour_data) > 1:
        df_clean = df_clean[~((df_clean['TOURNR'] == tour) &  (df_clean.index == tour_data.index[-1]))]

df_soll_final = df_clean.copy()  # Finaler Data frame

print(f"Letzte Stopps entfernt für {len(kreis_touren)} Touren")

Letzte Stopps entfernt für 236 Touren


In [72]:
df_soll_final

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
5,H/42375,OD-TS8050,"Szmajdewicz, Marcin",H/42375,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
...,...,...,...,...,...,...,...,...,...,...
1849,H/42754,WL-PL431,"Sowa, Mariusz",H/42754,2026-04-07 09:05:00,2026-04-07 09:35:00,9.98213,53.53497,0 days 00:30:00,2026-04-07
1851,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:00:00,2026-04-07 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-04-07
1852,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 06:57:00,2026-04-07 07:27:00,10.09782,53.53610,0 days 00:30:00,2026-04-07
1853,H/42758,PI-EN 444,"Teme, Abdoulaye",H/42758,2026-04-07 07:53:00,2026-04-07 08:23:00,9.98632,53.52348,0 days 00:30:00,2026-04-07


Beispiel einer gecleanten soll Tour

In [73]:
df_soll_final[df_soll_final['TOURNR'] == "H/42460"]

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
449,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 05:00:00,2026-03-10 06:45:00,9.99189,53.66737,0 days 01:45:00,2026-03-10
450,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 06:56:00,2026-03-10 07:21:00,10.04632,53.68627,0 days 00:25:00,2026-03-10
451,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 08:03:00,2026-03-10 08:28:00,10.37411,53.80550,0 days 00:25:00,2026-03-10
452,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 08:38:00,2026-03-10 09:03:00,10.40753,53.79811,0 days 00:25:00,2026-03-10
453,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 09:31:00,2026-03-10 09:56:00,10.23939,53.67382,0 days 00:25:00,2026-03-10
454,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 10:33:00,2026-03-10 10:58:00,10.19781,53.48664,0 days 00:25:00,2026-03-10
455,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 11:26:00,2026-03-10 11:51:00,10.01992,53.53592,0 days 00:25:00,2026-03-10
456,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 12:16:00,2026-03-10 12:41:00,10.11836,53.58145,0 days 00:25:00,2026-03-10
457,H/42460,OD-TS8041,"Jorczik, Christian",H/42460,2026-03-10 12:51:00,2026-03-10 13:16:00,10.11864,53.60523,0 days 00:25:00,2026-03-10


## Ist-Daten Formularbeschreibung pivotisieren
* Ziel: Mergen von Soll- und Ist-Daten
* Problem: in Soll-Daten gibt es eine Ankunft und eine Abfahrt Spalte, in Ist-Daten sind dies 2 Zeilen wo in der Formularbeschreibung 'Ankunft Kunde' oder 'Abfahrt Kunde' steht
* Lösung: aus 2 Zeilen eine Zeile machen mit einer Spalte Ankunft und einer Spalte Abfahrt statt Formularbeschreibung

### Entfernen der Zeilen ohne Tournummer (261 Zeilen)

In [74]:
df_ist_merged_clean = df_ist_merged[df_ist_merged.TOURNR.notna()]
df_ist_merged_clean

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 11:49:13,208,0,502860.415,0.0,0.0,"53,51717","9,96982","DE - 20457 Hamburg (Hamburg-Mitte), Köhlbrandb...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2871,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 11:59:39,0,0,502863.510,0.0,0.0,"53,52261","9,98925","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2872,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 12:11:29,0,0,502863.755,0.0,0.0,"53,5228","9,98903","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2873,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 12:37:22,0,0,502869.170,0.0,0.0,"53,5341","9,98237","DE - 20457 Hamburg (Hamburg-Mitte), Worthdamm 32",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432


### Entfernen von Mehrfachzeilen
* in kürzester Zeit (Sekunden Unterschiede) gibt es manchmal 2 mal Abfahrt Kunde hintereinander oder es gibt sowohl Ankunft als auch Abfahrt mehrmals am gleichen Ort, weil der Fahrer z.B. zu oft gedrückt hat -> Zeilen müssen reduziert werden -> bei gleichem KM-Stand wird nur eine Ankunft (früheste Zeit) und eine Abfahrt (späteste Zeit) Zeile behalten

Es gibt keine fehlenden Werte im KM-Stand und beim Blick auf eine bestimmte Tour scheint der KM-Stand auch zuverlässig zu sein -> Wir können die Spalte werden

In [75]:
df_ist_merged_clean[df_ist_merged_clean['KM-Stand'].isna()]

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR


Checken, ob es denselben KM-Stand für verschiedene Touren gibt

In [76]:
km = df_ist_merged_clean.groupby('KM-Stand').nunique().TOURNR
km

KM-Stand
0.000         44
143489.260     1
143493.015     1
143496.750     1
143500.460     1
              ..
808354.435     1
808360.995     1
808978.565     1
809290.085     1
809290.845     1
Name: TOURNR, Length: 1257, dtype: int64

In [77]:
# wie viele Zeilen gibt es mit KM-Stand 0
df_ist_merged_clean[df_ist_merged_clean['KM-Stand'] == 0].count()

Fahrer PIN              686
Fahrername              686
Formularnummer          686
Formularbeschreibung    686
Meldungszeit            686
Richtung                686
Geschwindigkeit         686
KM-Stand                686
Gesamtverbrauch         686
Tankfüllstand           686
Breitengrad             686
Längengrad              686
Position                686
Fahrzeug                686
DATUM                   686
KENNZ_CLEAN             686
TOURNR                  686
dtype: int64

In [78]:
# pro KM-Stand die Anzahl der Touren, die nicht 1 sind
km.where(lambda x: x!=1).dropna()

KM-Stand
0.000         44.0
179697.150     2.0
188279.015     2.0
195058.785     2.0
315830.410     2.0
317596.015     2.0
Name: TOURNR, dtype: float64

Ich habe alle diese Kilometerstände darauf gecheckt, dass es nicht 2 Touren an einem Tag sind

In [79]:
df_ist_merged_clean[df_ist_merged_clean['KM-Stand'] == 317596.015] 

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
547,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-04-01 15:28:35,0,0,317596.015,0.0,0.0,"53,66769","9,99274","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-04-01,OD-TS8041,H/42692
548,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-04-01 15:28:36,0,0,317596.015,0.0,0.0,"53,66769","9,99274","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-04-01,OD-TS8041,H/42692
549,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-04-02 05:01:51,0,0,317596.015,0.0,0.0,"53,66749","9,99298","DE - 22419 Hamburg (Langenhorn), Essener Straße 4",OD-TS 8041 - Fahrzeugdetails,2026-04-02,OD-TS8041,H/42682
550,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-04-02 05:01:54,0,0,317596.015,0.0,0.0,"53,66749","9,99298","DE - 22419 Hamburg (Langenhorn), Essener Straße 4",OD-TS 8041 - Fahrzeugdetails,2026-04-02,OD-TS8041,H/42682


* KM-Stand 0 gibt es in 686 Zeilen -> diese Zeilen können wir nicht reduzieren mithilfe des KM-Standes
* ansonsten gibt es noch einige Kilometerstände, die in 2 Touren vorkommen, aber da hat das Auto mit dem KM-Stand die Tour am Tag beendet und am nächsten Tag eine neue Tour logischerweise mit dem gleichen Kilometerstand gestartet -> neuer Tag -> keinen Einfluss auf unsere Daten 

### Hier werden Mehrfachzeilen entfernt

In [80]:
# Kopie zur Sicherheit
df = df_ist_merged_clean.copy()

# Meldungszeit als Timestamp parsen
df['Meldungszeit'] = pd.to_datetime(df['Meldungszeit'], dayfirst=True, errors='coerce')

# Für Ankunft Kunde: früheste Meldungszeit pro KM-Stand + DATUM
df_ankunft = (
    df[df['Formularbeschreibung'] == 'Ankunft Kunde']
      .sort_values(['KM-Stand', 'DATUM', 'Meldungszeit'], ascending=[True, True, True])
      .drop_duplicates(['KM-Stand', 'DATUM'], keep='first')
)

# Für Abfahrt Kunde: späteste Meldungszeit pro KM-Stand + DATUM
df_abfahrt = (
    df[df['Formularbeschreibung'] == 'Abfahrt Kunde']
      .sort_values(['KM-Stand', 'DATUM', 'Meldungszeit'], ascending=[True, True, False])
      .drop_duplicates(['KM-Stand', 'DATUM'], keep='first')
)

# Alle KM-Stand 0 Zeilen zusätzlich behalten, weil das nur fehlende Werte im KM-Stand sind und man damit keine Aussage treffen kann
km_zero = df_ist_merged_clean[df_ist_merged_clean['KM-Stand'] == 0].copy()

# Ergebnis zusammenführen
result = pd.concat([df_ankunft, df_abfahrt, km_zero])

# Ursprüngliche Reihenfolge anhand des ursprünglichen Index wiederherstellen
result = result.sort_index()

# Doppelte Zeilen entfernen, falls KM-Stand 0 zufällig auch in Ankunft/Abfahrt enthalten ist
result = result.drop_duplicates()

df_ist_merged_clean = result
df_ist_merged_clean

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
0,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 05:19:13,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
1,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:00:13,0,0,316550.025,0.0,0.0,"53,66757","9,9928","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
2,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:33:59,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
3,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-24 07:34:00,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
4,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-24 07:54:21,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 11:49:13,208,0,502860.415,0.0,0.0,"53,51717","9,96982","DE - 20457 Hamburg (Hamburg-Mitte), Köhlbrandb...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2871,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 11:59:39,0,0,502863.510,0.0,0.0,"53,52261","9,98925","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2872,8180.0,"Krüger, Alexander",1651,Abfahrt Kunde,2026-03-06 12:11:29,0,0,502863.755,0.0,0.0,"53,5228","9,98903","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432
2873,8180.0,"Krüger, Alexander",1650,Ankunft Kunde,2026-03-06 12:37:22,0,0,502869.170,0.0,0.0,"53,5341","9,98237","DE - 20457 Hamburg (Hamburg-Mitte), Worthdamm 32",TS859 - Fahrzeugdetails,2026-03-06,Plo-TS859,H/42432


In [81]:
# df_ist_merged_clean.to_csv('df_merged_clean.csv', index=False)

### Problem: Manchmal gibt es an einem Ort nur eine Abfahrtszeile und keine Ankunft (oder andersrum)
Es fehlt sozusagen ein Zeitstempel

In [82]:
df_ist_merged_clean[df_ist_merged_clean['KM-Stand'] == 188279.015]

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
638,8311.0,"Szmajdewicz, Marcin",1650,Ankunft Kunde,2026-03-30 17:19:05,0,0,188279.015,53581.0,48.4,"53,52341","9,98561","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",OD-TS 8050 - Fahrzeugdetails,2026-03-30,OD-TS8050,H/42645
639,8311.0,"Szmajdewicz, Marcin",1651,Abfahrt Kunde,2026-03-30 17:19:06,0,0,188279.015,53581.0,48.4,"53,52341","9,98561","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",OD-TS 8050 - Fahrzeugdetails,2026-03-30,OD-TS8050,H/42645
640,8311.0,"Szmajdewicz, Marcin",1650,Ankunft Kunde,2026-03-31 06:25:13,0,0,188279.015,53582.0,48.4,"53,52341","9,98562","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",OD-TS 8050 - Fahrzeugdetails,2026-03-31,OD-TS8050,H/42656
641,8311.0,"Szmajdewicz, Marcin",1651,Abfahrt Kunde,2026-03-31 06:25:14,0,0,188279.015,53582.0,48.4,"53,52341","9,98562","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",OD-TS 8050 - Fahrzeugdetails,2026-03-31,OD-TS8050,H/42656


In [83]:
# z.B. hier Zeile 13 (gucken in altes df um zu checken, dass das kein data cleaning fehler war)
df_ist_merged[df_ist_merged.TOURNR == 'H/42647']

,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR
503,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-30 05:18:19,0,0,317003.220,0.0,0.0,"53,6677","9,99287","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
504,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-30 06:17:22,0,0,317003.270,0.0,0.0,"53,66773","9,9932","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
505,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-30 06:59:54,131,0,317021.295,0.0,0.0,"53,55026","9,99017","DE - 20457 Hamburg (Hamburg-Mitte), Alter Wall 32",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
506,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-30 06:59:55,128,0,317021.295,0.0,0.0,"53,55026","9,99018","DE - 20457 Hamburg (Hamburg-Mitte), Alter Wall 32",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
507,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-30 07:28:23,0,0,317028.945,0.0,0.0,"53,57519","9,9127","DE - 22525 Hamburg (Bahrenfeld), Winsbergring 15",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
508,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-30 07:28:24,0,0,317028.945,0.0,0.0,"53,57519","9,9127","DE - 22525 Hamburg (Bahrenfeld), Winsbergring 15",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
509,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-30 08:37:24,0,0,317066.350,0.0,0.0,"53,61828","10,23092","DE - 22145 Braak, Waldweg 2",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
510,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-30 08:37:25,0,0,317066.350,0.0,0.0,"53,61828","10,23092","DE - 22145 Braak, Waldweg 2",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
511,8160.0,"Jorczik, Christian",1650,Ankunft Kunde,2026-03-30 09:24:06,0,0,317080.550,0.0,0.0,"53,67164","10,23997","DE - 22926 Ahrensburg, Manhagener Allee 10",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647
512,8160.0,"Jorczik, Christian",1651,Abfahrt Kunde,2026-03-30 09:24:07,0,0,317080.550,0.0,0.0,"53,67164","10,23997","DE - 22926 Ahrensburg, Manhagener Allee 10",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647


### Zeilen reduzieren und Tabelle pivotisieren
* Formularbeschreibung und Meldungszeit entfernen
* Stattdessen Spalten Ankunft und Abfahrt wie in Soll-Daten mit den jeweiligen Meldungszeiten und 2 Zeilen zu einem Stopp zusammenführen
* Dabei sollen bei gleichem KM-Stand, Ort (mit Geofencing Toleranz) und Datum die Zeilen zusammengeführt werden dürfen

In [84]:
# Parameter für die Toleranz
GEOFENCE_METERS = 30
KM_TOLERANCE = 0.2

df = df_ist_merged_clean.copy()

# Originale Reihenfolge merken
df["_row_order"] = np.arange(len(df))

# Neue Zeitspalten erzeugen
df["Abfahrt"] = df["Meldungszeit"].where(df["Formularbeschreibung"] == "Abfahrt Kunde")
df["Ankunft"] = df["Meldungszeit"].where(df["Formularbeschreibung"] == "Ankunft Kunde")

# Ursprüngliche Spaltenreihenfolge merken, ohne die später entfernten Spalten
original_cols = [c for c in df.columns if c not in ["Formularbeschreibung", "Meldungszeit", "_row_order"]]
km_pos = original_cols.index("KM-Stand")
final_order = original_cols[:km_pos] + original_cols[km_pos:]

# Numerische Hilfsspalten erzeugen
for col in ["Breitengrad", "Längengrad", "KM-Stand"]:
    df[col + "_num"] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .replace("nan", np.nan)
    )
    df[col + "_num"] = pd.to_numeric(df[col + "_num"], errors="coerce")

# Zeitspalte für interne Verarbeitung
df["Meldungszeit_dt"] = pd.to_datetime(df["Meldungszeit"], errors="coerce")

# Haversine-Distanz in Metern
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Cluster-ID erzeugen, aber in ursprünglicher Zeilenreihenfolge durchlaufen
cluster_ids = pd.Series(index=df.index, dtype="object")
cluster_first_row = {}

for datum, idx in df.groupby("DATUM", sort=False).groups.items():
    grp = df.loc[list(idx)].copy().sort_values("_row_order")
    grp_idx = grp.index.tolist()

    cluster_no = 0
    centers = []

    for i in grp_idx:
        row = df.loc[i]
        lat = row["Breitengrad_num"]
        lon = row["Längengrad_num"]
        km = row["KM-Stand_num"]

        assigned = False

        for c in centers:
            km_ok = (
                pd.isna(km) or pd.isna(c["km"]) or abs(km - c["km"]) <= KM_TOLERANCE
            )

            geo_ok = False
            if pd.notna(lat) and pd.notna(lon) and pd.notna(c["lat"]) and pd.notna(c["lon"]):
                geo_ok = haversine_m(lat, lon, c["lat"], c["lon"]) <= GEOFENCE_METERS

            if km_ok and geo_ok:
                cid = f"{datum}__{c['id']}"
                cluster_ids.loc[i] = cid
                cluster_first_row[cid] = min(cluster_first_row[cid], row["_row_order"])

                c["lat"] = np.nanmean([c["lat"], lat])
                c["lon"] = np.nanmean([c["lon"], lon])
                c["km"] = np.nanmean([c["km"], km])

                assigned = True
                break

        if not assigned:
            cluster_no += 1
            cid_short = f"c{cluster_no}"
            cid = f"{datum}__{cid_short}"
            cluster_ids.loc[i] = cid
            cluster_first_row[cid] = row["_row_order"]

            centers.append({
                "id": cid_short,
                "lat": lat,
                "lon": lon,
                "km": km
            })

df["cluster_id"] = cluster_ids

# Jetzt alte Spalten entfernen
df = df.drop(columns=["Formularbeschreibung", "Meldungszeit"])

# Aggregation
helper_cols = [
    "Breitengrad_num", "Längengrad_num", "KM-Stand_num",
    "Meldungszeit_dt", "_row_order", "cluster_id"
]

agg_dict = {
    col: "first"
    for col in df.columns
    if col not in helper_cols + ["Ankunft", "Abfahrt"]
}
agg_dict["Ankunft"] = "first"
agg_dict["Abfahrt"] = "first"
agg_dict["_row_order"] = "min"

df_stops = (
    df.groupby("cluster_id", dropna=False, as_index=False)
      .agg(agg_dict)
)

# Reihenfolge nach erster Originalzeile wiederherstellen
df_stops = df_stops.sort_values("_row_order", kind="stable").reset_index(drop=True)

# Hilfsspalte entfernen und gewünschte Spaltenreihenfolge setzen
df_stops = df_stops.drop(columns=["cluster_id", "_row_order"])
df_stops = df_stops[final_order]

# Optional speichern
# df_stops.to_csv("df_merged_clean_stops_geofence_ordered.csv", index=False)

df_stops.head()

,Fahrer PIN,Fahrername,Formularnummer,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR,Abfahrt,Ankunft
0,8160.0,"Jorczik, Christian",1650,74,0,316550.025,0.0,0.0,"53,66758","9,99266","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592,2026-03-24 07:00:13,2026-03-24 05:19:13
1,8160.0,"Jorczik, Christian",1650,0,0,316565.680,0.0,0.0,"53,58982","9,91377","DE - 22525 Hamburg (Eimsbüttel), Lederstraße 3...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592,2026-03-24 07:34:00,2026-03-24 07:33:59
2,8160.0,"Jorczik, Christian",1650,0,0,316568.980,0.0,0.0,"53,58884","9,87736","DE - 22547 Hamburg (Lurup), Elbgaustraße (Ring...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592,2026-03-24 07:54:22,2026-03-24 07:54:21
3,8160.0,"Jorczik, Christian",1650,117,0,316575.795,0.0,0.0,"53,55228","9,93833","DE - 22767 Hamburg (Altona-Altstadt), Große Be...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592,2026-03-24 08:27:18,2026-03-24 08:27:17
4,8160.0,"Jorczik, Christian",1650,0,0,316582.760,0.0,0.0,"53,53633","10,01815","DE - 20457 Hamburg (Hamburg-Mitte), Kirchenpau...",OD-TS 8041 - Fahrzeugdetails,2026-03-24,OD-TS8041,H/42592,2026-03-24 08:54:13,2026-03-24 08:54:10


In [85]:
df_stops[df_stops.TOURNR == 'H/42647']

,Fahrer PIN,Fahrername,Formularnummer,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR,Abfahrt,Ankunft
279,8160.0,"Jorczik, Christian",1650,0,0,317003.220,0.0,0.0,"53,6677","9,99287","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 06:17:22,2026-03-30 05:18:19
280,8160.0,"Jorczik, Christian",1650,131,0,317021.295,0.0,0.0,"53,55026","9,99017","DE - 20457 Hamburg (Hamburg-Mitte), Alter Wall 32",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 06:59:55,2026-03-30 06:59:54
281,8160.0,"Jorczik, Christian",1650,0,0,317028.945,0.0,0.0,"53,57519","9,9127","DE - 22525 Hamburg (Bahrenfeld), Winsbergring 15",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 07:28:24,2026-03-30 07:28:23
282,8160.0,"Jorczik, Christian",1650,0,0,317066.350,0.0,0.0,"53,61828","10,23092","DE - 22145 Braak, Waldweg 2",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 08:37:25,2026-03-30 08:37:24
283,8160.0,"Jorczik, Christian",1650,0,0,317080.550,0.0,0.0,"53,67164","10,23997","DE - 22926 Ahrensburg, Manhagener Allee 10",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 09:24:07,2026-03-30 09:24:06
284,8160.0,"Jorczik, Christian",1650,0,0,317105.330,0.0,0.0,"53,73133","10,02203","DE - 22844 Norderstedt, Oststraße 65",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 10:22:51,2026-03-30 10:22:50
285,8160.0,"Jorczik, Christian",1651,0,0,317113.850,0.0,0.0,"53,66773","9,99338","DE - 22419 Hamburg (Langenhorn), Essener Straße 4",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 10:54:27,NaT
286,8160.0,"Jorczik, Christian",1650,0,0,317144.030,0.0,0.0,"53,5341","10,22598","DE - 21509 Glinde, Holstenkamp 17",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 12:01:23,2026-03-30 12:01:22


In [86]:
#df_stops.to_csv('Ist-Daten.csv', index=False)

In [87]:
# df_soll_final.to_csv('Soll-Daten.csv', index=False)